# Custom DSPy programs and multi-stage patterns

This notebook covers Chapter 7's custom-program section: five composition patterns for combining DSPy modules, followed by a reusable custom `VerbalizedSampling` module. DSPy programs are just Python — there is no graph DSL or YAML config.

1. **Sequential pipeline** (`ContentModerator`)
2. **Conditional branching** (`SmartRouter`)
3. **Wrapping modules** (`ReliableClassifier`)
4. **Per-module model assignment** (`CostAwareAnalyzer`)
5. **Self-refinement loop** (`IterativeWriter`)
6. **Custom inference strategy** (`VerbalizedSampling`)

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## Pattern 1: sequential pipeline

The most common pattern. Each module's output feeds into the next module's input.


In [ ]:
import dspy

class ContentModerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classify = dspy.Predict(
            "content: str -> category: str, severity: str"
        )
        self.explain = dspy.ChainOfThought(
            "content: str, category: str, severity: str -> explanation: str, action: str"
        )

    def forward(self, content: str):
        classification = self.classify(content=content)
        decision = self.explain(
            content=content,
            category=classification.category,
            severity=classification.severity
        )
        return decision


## Pattern 2: conditional branching

Route to different modules based on input characteristics.


In [ ]:
class SmartRouter(dspy.Module):
    def __init__(self):
        super().__init__()
        self.router = dspy.Predict("query: str -> complexity: str")
        self.simple = dspy.Predict("query: str -> answer: str")
        self.complex = dspy.ChainOfThought("query: str -> answer: str")

    def forward(self, query: str):
        route = self.router(query=query)
        if route.complexity == "simple":
            return self.simple(query=query)
        return self.complex(query=query)


## Pattern 3: wrapping modules

Nest modules inside quality control wrappers like `BestOfN`.


In [ ]:
class ReliableClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        inner = dspy.ChainOfThought("text: str -> label: str")
        self.classify = dspy.BestOfN(
            module=inner, N=3,
            reward_fn=lambda args, pred: 1.0 if pred.label in ["positive", "negative", "neutral"] else 0.0,
            threshold=1.0
        )

    def forward(self, text: str):
        return self.classify(text=text)


## Pattern 4: per-module model assignment

Use a cheap model for easy steps, an expensive model for hard steps. `.set_lm()` lets you assign a different language model to each module.


In [ ]:
class CostAwareAnalyzer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.Predict("document: str -> entities: list[str]")
        self.analyze = dspy.ChainOfThought("entities: list[str] -> insights: str")

        # Cheap model for extraction, expensive model for analysis
        self.extract.set_lm(dspy.LM("openai/gpt-5.6-luna"))
        self.analyze.set_lm(dspy.LM("openai/gpt-4o"))

    def forward(self, document: str):
        extracted = self.extract(document=document)
        return self.analyze(entities=extracted.entities)


## Pattern 5: self-refinement loop

Generate, evaluate, and refine in a loop. This is the manual version of what `Refine` does automatically.


In [ ]:
class IterativeWriter(dspy.Module):
    def __init__(self):
        super().__init__()
        self.draft = dspy.ChainOfThought("brief: str -> draft_text: str")
        self.critique = dspy.ChainOfThought("brief: str, draft_text: str -> feedback: str, score: float")
        self.revise = dspy.ChainOfThought("brief: str, draft_text: str, feedback: str -> draft_text: str")

    def forward(self, brief: str):
        result = self.draft(brief=brief)
        for _ in range(3):
            review = self.critique(brief=brief, draft_text=result.draft_text)
            if review.score >= 0.9:
                break
            result = self.revise(brief=brief, draft_text=result.draft_text, feedback=review.feedback)
        return result


## Building a custom module: Verbalized Sampling

A custom module packages a prompting technique behind the same `dspy.Module` interface as DSPy's built-ins. This implementation asks for several typed candidates with estimated probabilities, then sorts them from most to least likely.


In [ ]:
import dspy
from pydantic import BaseModel, Field

class Candidate(BaseModel):
    text: str = Field(description="The response only, with no explanation or extra text.")
    probability: float = Field(
        description="Estimated probability from 0.0 to 1.0 of this response, "
                    "relative to the full distribution of possible responses."
    )

class VerbalizedSamplingSignature(dspy.Signature):
    """Generate several candidate responses to the prompt, each tagged with the
    probability the model would assign it. Draw from the full distribution so the
    set spans likely and unlikely responses, not just the most typical one."""

    prompt: str = dspy.InputField()
    k: int = dspy.InputField(desc="How many candidates to return.")
    candidates: list[Candidate] = dspy.OutputField()

class VerbalizedSampling(dspy.Module):
    def __init__(self, k: int = 5):
        super().__init__()
        self.k = k
        self.generate = dspy.Predict(VerbalizedSamplingSignature)

    def forward(self, prompt: str) -> dspy.Prediction:
        result = self.generate(prompt=prompt, k=self.k)
        candidates = sorted(result.candidates, key=lambda c: c.probability, reverse=True)
        return dspy.Prediction(candidates=candidates)


The typed `Candidate` output means DSPy returns parsed Pydantic objects rather than free-form text.


In [ ]:
dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna", temperature=1.0))

vs = VerbalizedSampling(k=5)
for c in vs(prompt="Tell me a joke about coffee.").candidates:
    print(f"{c.probability:.2f}  {c.text}")
